Hey! This is meant to run on [Google Colab](https://colab.research.google.com/)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install Conda (Colab doesn't have it by default, and MFA needs it)
!pip install -q condacolab
import condacolab
condacolab.install()

Mounted at /content/drive
⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...


In [ ]:
# Install MFA and its dependencies
!conda install -y -c conda-forge montreal-forced-aligner openai-whisper

Streaming output truncated to the last 5000 lines.
libcusparse-12.5.10. | 199.2 MB  | :  39% 0.39013254300436107/1 [00:02<00:03,  5.14s/it]
libcublas-12.9.1.4   | 446.1 MB  | :  19% 0.19254618618655214/1 [00:02<00:09, 11.91s/it]



libmagma-2.9.0       | 528.8 MB  | :  16% 0.1597048332243717/1 [00:02<00:12, 14.60s/it] 


libcusolver-11.7.5.8 | 195.6 MB  | :  42% 0.41704839895107493/1 [00:02<00:03,  5.48s/it]

libcusparse-12.5.10. | 199.2 MB  | :  41% 0.4096666277033528/1 [00:02<00:03,  5.32s/it] 
libcublas-12.9.1.4   | 446.1 MB  | :  20% 0.20109289701600797/1 [00:02<00:09, 12.40s/it]



libmagma-2.9.0       | 528.8 MB  | :  17% 0.16664852162543134/1 [00:02<00:12, 14.99s/it]


libcusolver-11.7.5.8 | 195.6 MB  | :  44% 0.43549692062049244/1 [00:02<00:03,  5.62s/it]

libcusparse-12.5.10. | 199.2 MB  | :  43% 0.4285731112875175/1 [00:02<00:03,  5.63s/it]
libcublas-12.9.1.4   | 446.1 MB  | :  21% 0.2092543053080703/1 [00:02<00:10, 12.71s/it] 



libmagma-2.9.0       | 528.8 MB  | :  17% 0.1

In [ ]:
!mfa model download acoustic english_us_arpa
!mfa model download dictionary english_us_arpa

In [ ]:
# This cell is here because whisper was refusing to import
import sys
import subprocess

# 1. Verify exactly which Python is running this cell
print(f"Current Python executable: {sys.executable}")

# 2. Force pip to run from INSIDE this exact Python kernel
print("Brute-forcing Whisper installation into this specific kernel...")
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "openai-whisper", "--quiet"], check=True)

import whisper
print("Whisper successfully imported!")

Current Python executable: /usr/bin/python3.real
Brute-forcing Whisper installation into this specific kernel...
Whisper successfully imported! The ghost is busted.


# That's all like prep/download stuff! Now onto the good stuff!!

In [ ]:
# # uncomment and run this cell if you're doing more than one genre at a time!
# # stuff gets weird when transcripts stick around too long

# import shutil
# shutil.rmtree(LOCAL_WORKSPACE)

In [ ]:
import os
import subprocess
import whisper

GENRE = 'metal'
DRIVE_AUDIO_DIR = f"/content/drive/MyDrive/gtzan/{GENRE}"

LOCAL_WORKSPACE = "/content/mfa_workspace"
os.makedirs(LOCAL_WORKSPACE, exist_ok=True)

print("Loading Whisper model...")
model = whisper.load_model("medium")

emptyFiles = []

# --- PROCESS FILES ---
print("Starting transcription...")
for filename in os.listdir(DRIVE_AUDIO_DIR):
    if filename.lower().endswith(".wav"): # MFA strictly expects WAV files

        drive_audio_path = os.path.join(DRIVE_AUDIO_DIR, filename)
        base_name = os.path.splitext(filename)[0]
        local_audio_path = os.path.join(LOCAL_WORKSPACE, f"{base_name}.wav")

        print(f"Transcribing: {filename}...")

        # 0. Downsample to 16k mono and move to local workspace
        # -ar 16000 (sample rate), -ac 1 (mono)
        subprocess.run([
            'ffmpeg', '-i', drive_audio_path,
            '-ar', '16000', '-ac', '1',
            local_audio_path, '-y', '-loglevel', 'quiet'
        ])

        # 1. Transcribe the audio
        result = model.transcribe(local_audio_path, fp16=False)
        transcript = result["text"].strip()
        if transcript.strip(" ") == "":
          emptyFiles.append(filename)
        print(transcript)

        # 3. Create the .lab file for MFA
        lab_path = os.path.join(LOCAL_WORKSPACE, f"{base_name}.lab")
        with open(lab_path, "w", encoding="utf-8") as f:
            f.write(transcript)

print("Transcription and MFA prep complete!")
print("The following files were empty:", emptyFiles)

Loading Whisper model...
Starting transcription...
Transcribing: 052_reggae.wav...

Transcribing: 080_hiphop.wav...
You're right, land dog. Tilt your head back, let's finish the call. MCA with the bottle. D, Ross Kahn. And Rockets, nice with Charlie Chan. We're all the more ass. We don't mind shivits wherever we go. We bring the monkey with us. As a tree tree. My dear D. Double off with the bill, most definitely. I'm Drew Brass Monkey. And I rock well. I got a castle in Brooklyn. That's where I twerk. Brass Monkey. That funky monkey. Brass Monkey. Drunky. That funky monkey. We're past. We're past. Chicken and time. At any point. We're past. At any point. At any point.
Transcribing: 070_pop.wav...

Transcribing: 035_metal.wav...
You'll take my life, but I'll take yours too. You're fire musket, but I'll run you through. So when you're waiting for the next attack, You better stand, there's no turning back. The bill of swords, the times begin. But on this battlefield no one wins. The spell

In [ ]:
# cell removes empty stuff from the workspace directory
print(len(emptyFiles))
for wav in emptyFiles:
  fileToRemove = os.path.join(LOCAL_WORKSPACE, wav)
  try:
    os.remove(fileToRemove)
  except FileNotFoundError:
    pass
  base_name = wav.split('.')[0]
  fileToRemove = os.path.join(LOCAL_WORKSPACE, f"{base_name}.lab")
  try:
    os.remove(fileToRemove)
  except FileNotFoundError:
    pass


44


In [ ]:
# # Writing this cell to zip the whisper prep for easy download! :D

# !zip -r /content/mfa_workspace.zip /content/mfa_workspace/


zip error: Invalid command arguments (long option 'single_speaker' not supported)


In [ ]:
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/gtzan/MFA_output"
import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

!mfa align /content/mfa_workspace english_us_arpa english_us_arpa "{DRIVE_OUTPUT_DIR}" --clean --single_speaker

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
  91% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 91/100  [ 0:00:01 < -:--:-- , ? it/s ]
 INFO     Found 1 speaker across 91 files, average number of utterances per     
          speaker: 91.0                                                         
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91/91  [ 0:00:01 < 0:00:00 , ? it/s ]
 INFO     Generating MFCCs...                                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91/91  [ 0:00:08 < 0:00:00 , ? it/s ]
 INFO     Calculating CMVN...                                                   
 INFO     Generating final features...                                          
 100% ━━━━━━━━━━━━━━━━━━━━━━